# Module 7: Evals (Optional) (13 min)

> **Optional module.** You've already built and deployed a complete agent in Modules 1–5. This module adds automated evaluation on top of that same agent.

Run automated evaluations against the customer service agent — an **output eval** (is the response good?) and a **trajectory eval** (did the agent follow the right workflow?).

**Prerequisites:** Modules 1-4 completed

In [1]:
!pip install -q -r requirements.txt

# The evals framework calls asyncio.run() internally, which clashes with the
# event loop Jupyter already runs. nest_asyncio patches the loop so the
# synchronous run_evaluations() API works inside a notebook cell.
import nest_asyncio

nest_asyncio.apply()

---

## Part 1: Output Evaluation — Is the Response Good?

The `OutputEvaluator` uses LLM-as-a-judge to score agent responses against expected outputs.

An eval is only useful if it can tell a **good** answer from a **bad** one. To prove that,
we run the same test cases against two agents:

1. ❌ **A weak agent with no tools** — it has no way to look up real customer data, so it
   guesses. The judge should give it **low scores**.
2. ✅ **The real agent with tools** — it looks up the actual data. The judge should give it
   **high scores**.

Seeing the weak agent fail is the point: it shows the rubric is actually doing its job, not
just stamping everything 1.0.

In [2]:
from strands import Agent
from strands_evals import eval_task, Case, Experiment
from strands_evals.evaluators import OutputEvaluator
from customer_service_tools import lookup_customer, get_order_history, process_refund

SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise. Use the available tools to look up customer
information and process requests.

Important: Always verify the customer first, then check orders if needed."""


# The collapsed table shows "..." for the reason column. This helper prints the
# judge's reasoning for every case in plain text so you can see WHY each scored
# the way it did — that "why" is the most valuable part of an LLM-as-judge eval.
def print_reasons(report):
    for i, score in enumerate(report.scores):
        name = report.cases[i].get("name", f"case-{i}")
        mark = "✅" if report.test_passes[i] else "❌"
        reason = report.reasons[i] if i < len(report.reasons) else "(no reason returned)"
        print(f"\n{mark} {name}  (score {score:.2f})")
        print(f"   {reason}")


# Two agents under test: a weak one (no tools) and the real one (with tools).
@eval_task()
def weak_agent():
    """No tools — it can't look up real data, so it guesses. Expect LOW scores."""
    return Agent(
        system_prompt="You are a customer service agent. Answer as best you can.",
        callback_handler=None,
    )


@eval_task()
def good_agent():
    """Has tools — it looks up the real data. Expect HIGH scores."""
    return Agent(
        tools=[lookup_customer, get_order_history, process_refund],
        system_prompt=SYSTEM_PROMPT,
        callback_handler=None,
    )


# Test cases — the same questions for both agents
output_cases = [
    Case[str, str](
        name="order-status-check",
        input="I'm customer C-1001. Where is my USB-C Hub order?",
        expected_output="The USB-C Hub is shipped with tracking TRK-887766, estimated delivery 2025-05-06.",
    ),
    Case[str, str](
        name="delayed-order-empathy",
        input="I'm customer C-1002. My keyboard order is delayed and I'm frustrated!",
        expected_output="Acknowledge frustration, provide order status for the delayed mechanical keyboard with tracking info.",
    ),
    Case[str, str](
        name="unknown-customer",
        input="I'm customer C-9999. What are my orders?",
        expected_output="Inform the customer that no account was found with that ID and ask them to verify.",
    ),
]

# Define the evaluator rubric
output_evaluator = OutputEvaluator(
    rubric="""
    Evaluate the customer service response against the expected output:
    1. Accuracy — Does it contain the correct order/tracking details? Invented or
       missing details (e.g. a made-up tracking number) must score low.
    2. Tone — Is it professional and empathetic?
    3. Completeness — Does it fully address the customer's concern?

    Score 1.0 only if accuracy is correct AND tone and completeness are met.
    Score 0.5 if partially met (e.g. right tone but wrong/missing facts).
    Score 0.0 if the information is inadequate, invented, or incorrect.
    """,
    include_inputs=True,
)

print("❌ Weak agent (no tools) — expect LOW scores:")
weak_report = Experiment[str, str](cases=output_cases, evaluators=[output_evaluator]).run_evaluations(weak_agent)
# Use .display() (static render) in notebooks — .run_display() opens an
# interactive Rich view that blocks the cell waiting for input.
weak_report.display(include_actual_output=True)
print_reasons(weak_report)  # the judge's "why" for each case

print("\n✅ Real agent (with tools) — expect HIGH scores:")
good_report = Experiment[str, str](cases=output_cases, evaluators=[output_evaluator]).run_evaluations(good_agent)
good_report.display(include_actual_output=True)
print_reasons(good_report)

print(
    f"\n📊 No-tools agent: {weak_report.overall_score:.2f}"
    f"  vs  with-tools agent: {good_report.overall_score:.2f}"
)
print("The gap is the eval doing its job — it catches the agent that guesses.")

❌ Weak agent (no tools) — expect LOW scores:


Task was destroyed but it is pending!
task: <Task cancelling name='Task-8' coro=<_poll_cancel_signal() running at /usr/local/lib/python3.13/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>
Task was destroyed but it is pending!
task: <Task cancelling name='Task-198' coro=<_poll_cancel_signal() running at /usr/local/lib/python3.13/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>
Task was destroyed but it is pending!
task: <Task cancelling name='Task-536' coro=<_poll_cancel_signal() running at /usr/local/lib/python3.13/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.33           Pass Rate: 0.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                           Test Case Results                                            
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ index ┃ name                  ┃ evaluator       ┃ score ┃ test_pass ┃ reason ┃ input ┃ actual_output ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ▶ 0   │ order-status-check    │ OutputEvaluator │ 0.30  │ ❌        │ ...    │ ...   │ ...           │
├───────┼───────────────────────┼─────────────────┼───────┼───────────┼────────┼───────┼───────────────┤
│ ▶ 1   │ delayed-order-empathy │ OutputEvaluator │ 0.30  │ ❌        │ ...    │ ...   │ ...           │
├───────┼───────────────────────┼─────────────────┼───────┼───────────┼────────┼───────┼───────────────┤
│ ▶ 2   │ unknown-customer      │ OutputEvaluator │ 0.40  │ ❌        │ ...    │ ...   │ ...           │
└───────┴───────────────────────┴─────────────────┴───────┴───────────┴────────┴───────┴───────────────┘


❌ order-status-check  (score 0.30)
   The response completely fails on accuracy — it does not provide the correct tracking number (TRK-887766) or estimated delivery date (2025-05-06) that were expected. The agent claims it has no access to order information and redirects the customer elsewhere, which is factually inadequate given what the expected output shows should have been retrievable. Tone is professional and empathetic, and the response is structured well, but since the core factual information is entirely missing, it cannot score higher. A small partial credit is given for tone and appropriate structure, but the critical accuracy and completeness dimensions are both failed.

❌ delayed-order-empathy  (score 0.30)
   The response demonstrates good empathy and professional tone, but fails critically on accuracy and completeness. It provides no order status or tracking information for customer C-1002's delayed keyboard order, despite having the customer ID. The agent even asks the 

Task was destroyed but it is pending!
task: <Task cancelling name='Task-820' coro=<_poll_cancel_signal() running at /usr/local/lib/python3.13/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>
Task was destroyed but it is pending!
task: <Task cancelling name='Task-1052' coro=<_poll_cancel_signal() running at /usr/local/lib/python3.13/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>
Task was destroyed but it is pending!
task: <Task cancelling name='Task-1368' coro=<_poll_cancel_signal() running at /usr/local/lib/python3.13/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                           Test Case Results                                            
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ index ┃ name                  ┃ evaluator       ┃ score ┃ test_pass ┃ reason ┃ input ┃ actual_output ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ▶ 0   │ order-status-check    │ OutputEvaluator │ 1.00  │ ✅        │ ...    │ ...   │ ...           │
├───────┼───────────────────────┼─────────────────┼───────┼───────────┼────────┼───────┼───────────────┤
│ ▶ 1   │ delayed-order-empathy │ OutputEvaluator │ 1.00  │ ✅        │ ...    │ ...   │ ...           │
├───────┼───────────────────────┼─────────────────┼───────┼───────────┼────────┼───────┼───────────────┤
│ ▶ 2   │ unknown-customer      │ OutputEvaluator │ 1.00  │ ✅        │ ...    │ ...   │ ...           │
└───────┴───────────────────────┴─────────────────┴───────┴───────────┴────────┴───────┴───────────────┘


✅ order-status-check  (score 1.00)
   The output accurately matches all key facts from the expected output: tracking number TRK-887766 and estimated delivery May 6, 2025. Additional details (Order ID, order date, customer name) are consistent with a legitimate data lookup. Tone is professional and empathetic, and the response fully addresses the customer's concern about their order status.

✅ delayed-order-empathy  (score 1.00)
   The response accurately addresses customer C-1002 by name (Mike), provides specific and plausible order details (ORD-5390, Mechanical Keyboard, $149.99, order/delivery dates, tracking number TRK-776655) consistent with a data-backed system. The tone is empathetic and professional, directly acknowledging the frustration. The response is complete — it acknowledges the delay, provides full order/tracking info, and offers actionable next steps. All three rubric criteria (accuracy, tone, completeness) are fully met.

✅ unknown-customer  (score 1.00)
   **Accuracy

---

## Part 2: Trajectory Evaluation — Did It Follow the Workflow?

The `TrajectoryEvaluator` checks that the agent called tools in the **correct order**. For a
refund, the policy is: look up the customer → check their order history → only then process
the refund.

Again we compare two agents to prove the eval works:

1. ❌ **A naive agent** told to "process refunds immediately" — it skips the verification
   steps. The trajectory eval should **fail** it.
2. ✅ **A steered agent** with an explicit ordered workflow — it follows the policy. The
   trajectory eval should **pass** it.

This is how you catch a missing guardrail before it reaches production.

In [3]:
from strands_evals.evaluators import TrajectoryEvaluator
from strands_evals.extractors import tools_use_extractor
from strands_evals.types import TaskOutput

# Two system prompts: one that skips verification, one that enforces the workflow.
NAIVE_PROMPT = """You are a fast customer service agent. When a customer asks for a refund,
process it right away with process_refund. Don't waste time on extra lookups."""

STEERED_PROMPT = """You are a customer service agent. When processing refunds, you MUST:
1. First look up the customer
2. Then check their order history
3. Only then process the refund
Always follow this exact order."""


def run_with(prompt):
    """Build a task function that runs an agent with the given system prompt
    and captures the tools it called."""

    def task(case: Case) -> TaskOutput:
        agent = Agent(
            tools=[lookup_customer, get_order_history, process_refund],
            system_prompt=prompt,
            callback_handler=None,
        )
        response = agent(case.input)
        trajectory = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)
        return TaskOutput(output=str(response), trajectory=trajectory)

    return task


# Cases with expected tool sequences
trajectory_cases = [
    Case[str, str](
        name="refund-workflow",
        input="Customer C-1001 wants a refund for order ORD-5521 ($79.99). Process it now.",
        expected_trajectory=["lookup_customer", "get_order_history", "process_refund"],
    ),
    Case[str, str](
        name="info-lookup-only",
        input="Look up customer C-1001 and tell me their order history.",
        expected_trajectory=["lookup_customer", "get_order_history"],
    ),
    Case[str, str](
        name="customer-lookup-only",
        input="Look up customer C-1002's account info.",
        expected_trajectory=["lookup_customer"],
    ),
]

# Create trajectory evaluator
trajectory_evaluator = TrajectoryEvaluator(
    rubric="""
    Evaluate whether the agent followed the expected tool sequence:
    - The expected tools should appear in order (extra tools in between are OK).
    - Score 1.0 if the expected sequence is followed correctly.
    - Score 0.5 if tools are called but in wrong order.
    - Score 0.0 if expected tools are missing entirely.
    """,
    include_inputs=True,
)

# Give evaluator context about available tools
sample_agent = Agent(tools=[lookup_customer, get_order_history, process_refund])
tool_descriptions = tools_use_extractor.extract_tools_description(sample_agent, is_short=True)
trajectory_evaluator.update_trajectory_description(tool_descriptions)

print("❌ Naive agent (skips verification) — expect a FAILED refund trajectory:")
naive_report = Experiment[str, str](cases=trajectory_cases, evaluators=[trajectory_evaluator]).run_evaluations(run_with(NAIVE_PROMPT))
# .display() renders statically; .run_display() would block the cell (see Part 1).
naive_report.display(include_actual_trajectory=True, include_expected_trajectory=True)
print_reasons(naive_report)  # see WHY the refund case failed (defined in Part 1)

print("\n✅ Steered agent (enforced workflow) — expect all trajectories to PASS:")
steered_report = Experiment[str, str](cases=trajectory_cases, evaluators=[trajectory_evaluator]).run_evaluations(run_with(STEERED_PROMPT))
steered_report.display(include_actual_trajectory=True, include_expected_trajectory=True)
print_reasons(steered_report)

print(
    f"\n📊 Naive agent: {naive_report.overall_score:.2f}"
    f"  vs  steered agent: {steered_report.overall_score:.2f}"
)
print("The steering handlers are what close that gap — and the eval is what proves it.")

❌ Naive agent (skips verification) — expect a FAILED refund trajectory:


Task was destroyed but it is pending!
task: <Task cancelling name='Task-1520' coro=<_poll_cancel_signal() running at /usr/local/lib/python3.13/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.67           Pass Rate: 0.6666666666666666                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                 Test Case Results                                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ index ┃ name           ┃ evaluator      ┃ score ┃ test_pass ┃ reason ┃ input ┃ actual_traject… ┃ expected_traj… ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ ▶ 0   │ refund-workfl… │ TrajectoryEva… │ 0.00  │ ❌        │ ...    │ ...   │ ...             │ ...            │
├───────┼────────────────┼────────────────┼───────┼───────────┼────────┼───────┼─────────────────┼────────────────┤
│ ▶ 1   │ info-lookup-o… │ TrajectoryEva… │ 1.00  │ ✅        │ ...    │ ...   │ ...             │ ...            │
├───────┼────────────────┼────────────────┼───────┼───────────┼────────┼───────┼─────────────────┼────────────────┤
│ ▶ 2   │ customer-look… │ TrajectoryEva… │ 1.00  │ ✅        │ ...    │ ...   │ ...             │ ...            │
└───────┴────────────────┴────────────────┴───────┴───────────┴────────┴───────┴─────────────────┴────────────────┘


❌ refund-workflow  (score 0.00)
   The agent only called `process_refund`, completely skipping the required `lookup_customer` and `get_order_history` steps. The expected sequence required all three tools in order. Since two of the three expected tools are missing entirely, and the rubric specifies a score of 0.0 when expected tools are missing entirely, the score is 0.0. The output appears correct, but the agent bypassed necessary verification steps before processing the refund.

✅ info-lookup-only  (score 1.00)
   The actual trajectory follows the expected tool sequence exactly: `lookup_customer` is called first (with customer_id 'C-1001'), followed by `get_order_history` (also with customer_id 'C-1001'). Both tools executed successfully without errors. The order matches the expected sequence perfectly, and the output accurately reflects all the data returned by the tools. This is a full score.

✅ customer-lookup-only  (score 1.00)
   The actual trajectory shows that `lookup_customer

Task was destroyed but it is pending!
task: <Task cancelling name='Task-2071' coro=<_poll_cancel_signal() running at /usr/local/lib/python3.13/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>
Task was destroyed but it is pending!
task: <Task cancelling name='Task-2598' coro=<_poll_cancel_signal() running at /usr/local/lib/python3.13/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                 Test Case Results                                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ index ┃ name           ┃ evaluator      ┃ score ┃ test_pass ┃ reason ┃ input ┃ actual_traject… ┃ expected_traj… ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ ▶ 0   │ refund-workfl… │ TrajectoryEva… │ 1.00  │ ✅        │ ...    │ ...   │ ...             │ ...            │
├───────┼────────────────┼────────────────┼───────┼───────────┼────────┼───────┼─────────────────┼────────────────┤
│ ▶ 1   │ info-lookup-o… │ TrajectoryEva… │ 1.00  │ ✅        │ ...    │ ...   │ ...             │ ...            │
├───────┼────────────────┼────────────────┼───────┼───────────┼────────┼───────┼─────────────────┼────────────────┤
│ ▶ 2   │ customer-look… │ TrajectoryEva… │ 1.00  │ ✅        │ ...    │ ...   │ ...             │ ...            │
└───────┴────────────────┴────────────────┴───────┴───────────┴────────┴───────┴─────────────────┴────────────────┘


✅ refund-workflow  (score 1.00)
   The agent followed the expected tool sequence exactly: (1) `lookup_customer` to verify customer C-1001, (2) `get_order_history` to confirm order ORD-5521 and its details, and (3) `process_refund` to execute the $79.99 refund. All three tools appear in the correct order with no missing steps or out-of-order calls. The output accurately reflects the tool results.

✅ info-lookup-only  (score 1.00)
   The agent followed the expected tool sequence exactly: (1) `lookup_customer` was called first to retrieve customer details for C-1001, and (2) `get_order_history` was called second to retrieve the order history — matching the expected order perfectly. No tools are missing and no tools are out of order. The output also correctly surfaces all data returned by the tools.

✅ customer-lookup-only  (score 1.00)
   The actual trajectory correctly uses the `lookup_customer` tool with the appropriate `customer_id: C-1002` parameter, matching the expected tool sequen

---

## 🎯 Try It Yourself

You've seen the eval catch a weak agent and a missing guardrail. Now extend it:

- Add a case where **no customer ID is given** — the agent should ask, not guess, so the
  expected trajectory is empty `[]`.
- Or tweak the `NAIVE_PROMPT` above and watch the scores move. That feedback loop — change
  the agent, rerun the eval, compare the numbers — is exactly how you harden an agent.

In [7]:
# Challenge: Add a case where no customer ID is given
# Expected trajectory should be empty [] — the agent should ask, not guess

# new_case = Case[str, str](
#     name="missing-customer-id",
#     input="I want to return something I bought last week.",
#     expected_trajectory=[],
# )

# Your code here...

---

## What's Next

You've validated the agent works correctly — and you already deployed it back in **Module 5: Deploy**. That completes the workshop. To take the agent to production operations (managed Gateway tools, memory, and observability), see the follow-on workshop.